# MAIE5102 Assignment1:

### Contact TA:
Shuling Zhao: szhaoax@cse.ust.hk
Yuxin Wang: ywangom@cse.ust.hk

You need to install the matplotlib package in advance, try this command:
`
pip3 install matplotlib
`
or
`
conda install matplotlib
`

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import csv
import random

In [ ]:
def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True


def AccuarcyCompute(pred: torch.Tensor, label: torch.Tensor):
    pred = pred.cpu().data.numpy()
    label = label.cpu().data.numpy()
    test_np = (np.argmax(pred, 1) == label)
    test_np = np.float32(test_np)
    return np.mean(test_np)


def plot_fig(Y: list, title: str, dir: str, X=None, x_label=None):
    if X is None:
        plt.plot(Y)
    else:
        plt.plot(X, Y)
    if 'train' in title or 'loss' in title:
        plt.ylabel('loss')
    else:
        plt.ylabel('accuracy')
    if x_label:
        plt.xlabel(x_label)
    else:
        plt.xlabel('epoch')
    plt.title(title)
    plt.savefig(dir)
    plt.show()


def plot_decision_boundary(model, dataset_test, labels, layer_num=None, 
                                     title=None, color_map='coolwarm',
                                     name='decision_boundary_'):
    if torch.is_tensor(dataset_test):
        dataset = dataset_test.detach().cpu().numpy()
    else:
        dataset = dataset_test
    
    if torch.is_tensor(labels):
        labels = labels.cpu().numpy()
    
    x_min, x_max = dataset[:, 0].min() - 1, dataset[:, 0].max() + 1
    y_min, y_max = dataset[:, 1].min() - 1, dataset[:, 1].max() + 1
    h = 0.02  # step size in the mesh
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    grid_points = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
    model.eval()  # Set model to evaluation mode
    with torch.no_grad():
        predictions = model(grid_points)
        Z = torch.argmax(predictions, dim=1).cpu().numpy()
    Z = Z.reshape(xx.shape)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    color_map_obj = plt.get_cmap(color_map)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=color_map_obj)    
    scatter = ax.scatter(dataset[:, 0], dataset[:, 1], 
                        c=labels, cmap=color_map_obj,
                        edgecolors='black', linewidth=1.5, s=50)
    
    plt.xlabel('feature 1', fontsize=12)
    plt.ylabel('feature 2', fontsize=12)
    
    if layer_num:
        plt.title(f'Decision boundary with {layer_num} hidden units', fontsize=14)
        plt.savefig(f'{name}{layer_num}.png', dpi=150, bbox_inches='tight')
    elif title:
        plt.title(title, fontsize=14)
        plt.savefig(f'{title}.png', dpi=150, bbox_inches='tight')
    else:
        plt.title('Decision Boundary', fontsize=14)
    
    plt.tight_layout()
    plt.show()


def load_data(dataset: str):
    path_list = [dataset + '_train.csv', dataset + '_valid.csv', dataset + '_test.csv']
    data_list_total, label_list_total = [], []
    for path in path_list:
        data_list, label_list = [], []
        with open(path, 'r') as file:
            reader = csv.reader(file)
            for idx, row in enumerate(reader):
                data_list.append([float(x) for x in row[:-1]])
                label_list.append(int(row[-1]))
        data_list_total.append(data_list)
        label_list_total.append(label_list)
    return data_list_total, label_list_total


## Problem 2: 2 cluster

In [ ]:
setup_seed(5102)

data_list, label_list = load_data('2_cluster')
train_data = data_list[0]
train_label = label_list[0]
valid_data = data_list[1]
valid_label = label_list[1]
test_data = data_list[2]
test_label = label_list[2]
train_data.extend(valid_data)
train_label.extend(valid_label)
train_input = torch.tensor(train_data)
train_label = torch.tensor(train_label)
test_input = torch.tensor(test_data)
test_label = torch.tensor(test_label)

Modify the functions below:

In [ ]:
def build_1_layer_mlp(nbr_hidden_unit):
    # Your code here
    return mlp


In [ ]:
def train_mlp():
    epochs = 30
    loss_list, test_acc = [], []
    lossfunc = torch.nn.CrossEntropyLoss()
    hidden_unit_list = [2, 5, 10, 50, 100, 500, 1000, 5000]
    for hidden_unit in hidden_unit_list:
        mlp = build_1_layer_mlp(hidden_unit)
        optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)
        for i_epoch in range(epochs):
            optimizer.zero_grad()
            outputs = mlp(train_input)
            loss = lossfunc(outputs, train_label)
            loss.backward()
            optimizer.step()
        with torch.no_grad():
            test_outputs = mlp(test_input)
            acc = AccuarcyCompute(test_outputs, test_label)
            test_acc.append(acc)
            print('Accuracy :{}'.format(acc))
            predicted_label = np.argmax(test_outputs.cpu().data.numpy(), 1)
        plot_decision_boundary(mlp, test_input, test_label, layer_num=hidden_unit)
    plot_fig(test_acc, 'Test acc for MLPs', '2_cluster_test_acc.png', [str(i) for i in hidden_unit_list], x_label='hidden units')


train_mlp()

## Problem 3 : Wine

In [ ]:
setup_seed(5102)
data_list, label_list = load_data('wine')
train_data = data_list[0]
train_label = label_list[0]
valid_data = data_list[1]
valid_label = label_list[1]
test_data = data_list[2]
test_label = label_list[2]
train_input = torch.tensor(train_data)
train_label = torch.tensor(train_label)
valid_input = torch.tensor(valid_data)
valid_label = torch.tensor(valid_label)
test_input = torch.tensor(test_data)
test_label = torch.tensor(test_label)

In [ ]:
def build_mlp():
    # Your code here
    return mlp


In [ ]:
def train_mlp():
    # Your code here

train_mlp()